In [1]:
pip install python-dotenv

In [2]:
from dotenv import load_dotenv
load_dotenv()

False

In [5]:
pip install uv

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 53.4 MB/s eta 0:00:00


In [6]:
!uv init

Initialized project `content`


In [7]:
!uv add openai langchain-openai langchain-google-genai langgraph jinja2 json-repair tavily

Using CPython 3.12.13 interpreter at: /usr/bin/python3
Creating virtual environment at: .venv
Resolved 63 packages in 1.27s
Prepared 61 packages in 1.99s
Installed 61 packages in 113ms
 + aiohappyeyeballs==2.6.1
 + aiohttp==3.13.4
 + aiosignal==1.4.0
 + annotated-types==0.7.0
 + anyio==4.13.0
 + attrs==26.1.0
 + certifi==2026.2.25
 + cffi==2.0.0
 + charset-normalizer==3.4.6
 + cryptography==46.0.6
 + distro==1.9.0
 + filetype==1.2.0
 + frozenlist==1.8.0
 + google-auth==2.49.1
 + google-genai==1.69.0
 + h11==0.16.0
 + httpcore==1.0.9
 + httpx==0.28.1
 + idna==3.11
 + jinja2==3.1.6
 + jiter==0.13.0
 + json-repair==0.58.7
 + jsonpatch==1.33
 + jsonpointer==3.1.1
 + langchain-core==1.2.23
 + langchain-google-genai==4.2.1
 + langchain-openai==1.1.12
 + langgraph==1.1.3
 + langgraph-checkpoint==4.0.1
 + langgraph-prebuilt==1.0.8
 + langgraph-sdk==0.3.12
 + langsmith==0.7.23
 + markupsafe==3.0.3
 + multidict==6.7.1
 + openai==2.30.0
 + orjson==3.11.7
 + ormsgpack==1.12.2
 + packaging==26.0
 +

In [9]:
from openai import OpenAI
import json
import os


def check_delivery_status(order_id: str):
    return {
        "order_id": order_id,
        "order_status": "In delivery",
        "order_last_location": "Sedang disortir di DC cakung",
        "ETA": "31-12-2025",
    }


def check_shopping_cart(user_id: str):
    return {
        "user_id": user_id,
        "items_in_cart": [
            {
                "item_id": 1,
                "item_name": "babibas running shoes",
                "item_type": "shoes",
                "item_details": {"size": 48, "color": "bright pink"},
            },
            {
                "item_id": 22,
                "item_name": "bortusten performance socks",
                "item_type": "socks",
                "item_details": {"size": "XL", "color": "bright yellow"},
            },
            {
                "item_id": 111,
                "item_name": "becs anti-slip shoe lace",
                "item_type": "shoe_accessories",
                "item_details": {"color": "cyan"},
            },
        ],
    }


# ganti base URL kalau menggunakan provider lain
MODEL_NAME = "gemini-2.5-flash"
client = OpenAI(
    api_key=os.getenv("API_KEY_LLM"),
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/",
)


# Definisi tool
tools = [
    {
        "type": "function",
        "function": {
            "name": "check_delivery_status",
            "description": "Check the delivery status based on order ID.",
            "parameters": {
                "type": "object",
                "properties": {
                    "order_id": {"type": "string"},
                },
                "required": ["order_id"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "check_shopping_cart",
            "description": "Retrieve the user's shopping cart based on user ID.",
            "parameters": {
                "type": "object",
                "properties": {
                    "user_id": {"type": "string"},
                },
                "required": ["user_id"],
            },
        },
    },
]

tools_dict = {
    "check_delivery_status": check_delivery_status,
    "check_shopping_cart": check_shopping_cart,
}

SYSTEM_PROMPT = """\
Act as a friendly shopping assistant. You will be provided with tools to help you answer the user's queries.
When a tool is not available to help a user's request, suggest to contact a human instead.
Do not make up answers, if a user does not provide the required information, ask them.
Answer should be in fully natural language. It's okay to add small formatting.
Answer in Indonesian by default unless the user asks in english.
"""


# By default menggunakan gemini. Ganti nama model kalau menggunakan provider lain
def chat(query):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": query},
    ]

    # Pemanggilan pertama
    response = client.chat.completions.create(
        model=MODEL_NAME,
        messages=messages,
        tools=tools,
        tool_choice="auto",
    )

    msg = response.choices[0].message

    # Kalau model memanggil tool, responnya akan berisi
    # tool call
    if msg.tool_calls:
        messages.append(msg)
        for call in msg.tool_calls:
            try:
                ## memetakan nama tool ke fungsinya
                tool_func = tools_dict[call.function.name]
                ## print untuk melihat tool apa yang dipanggil untuk chatnya
                print(f"Tool {call.function.name} was called.")
                args = json.loads(call.function.arguments)

                # LLM cuma memberikan parameter dan menentukan tool yang dipanggil
                # implementasi toolnya dipanggil dalam kode
                results = tool_func(**args)

                # Cara khusus untuk mengembalikan hasil tool call
                # ke model untuk dipakai menghasilkan jawaban
                messages.append(
                    {
                        "role": "tool",
                        "tool_call_id": call.id,
                        "content": json.dumps(results),
                    }
                )
            # error handling kalau tool tidak ditemukan (LLM halusinasi nama tool)
            except KeyError:
                messages.append(
                    {
                        "role": "tool",
                        "tool_call_id": call.id,
                        "content": json.dumps(
                            {
                                "status": "error",
                                "message": f"Tool {call.function.name} does not exist.",
                            }
                        ),
                    }
                )
            # error handling kalau ada error lain
            # idealnya setiap jenis error ada cara handling masing-masing
            # tapi disatukan dulu untuk mempermudah contoh
            except Exception as ex:
                messages.append(
                    {
                        "role": "tool",
                        "tool_call_id": call.id,
                        "content": json.dumps(
                            {
                                "status": "error",
                                "message": f"Tool {call.function.name} errored with an unknown error: {ex}."
                                "Give generic response to the user that something is wrong.",
                            }
                        ),
                    }
                )
        # lanjut menghasilkan jawaban
        final = client.chat.completions.create(model=MODEL_NAME, messages=messages)
        return final.choices[0].message.content

    # Kalau modelnya nggak manggil tool,
    # langsung berikan saja jawabannya
    return msg.content

print(">> Pemanggilan 1, perlu tool check status pengiriman <<")
print(chat("Halo, saya ingin mengecek status pengiriman pesanan saya dengan ID 1112231233"))
print("====== separator ======")
print(">> Pemanggilan 2, perlu tool check keranjang belanja <<")
print(
    chat("Halo, saya ingin mengecek keranjang belanja saya dengan user ID Ivan#19827")
)
print("====== separator ======")
print(">> Pemanggilan 3, tidak perlu tool apapun, cuma pertanyaan biasa <<")
print(
    chat("Halo, apa kabar? Siapa kamu?")
)

>> Pemanggilan 1, perlu tool check status pengiriman <<
Tool check_delivery_status was called.
Baik, saya sudah mengecek status pesanan Anda dengan ID **1112231233**.

Status pesanan Anda saat ini adalah: **Dalam Pengiriman**.
Lokasi terakhirnya adalah: **Sedang disortir di DC Cakung**.
Estimasi waktu tiba (ETA): **31 Desember 2025**.

Semoga informasi ini membantu! Ada hal lain yang bisa saya bantu?
====== separator ======
>> Pemanggilan 2, perlu tool check keranjang belanja <<
Tool check_shopping_cart was called.
Baik Ivan, saya sudah berhasil mengecek keranjang belanja Anda. Berikut adalah isinya:

*   **babibas running shoes** (Ukuran: 48, Warna: bright pink)
*   **bortusten performance socks** (Ukuran: XL, Warna: bright yellow)
*   **becs anti-slip shoe lace** (Warna: cyan)

Apakah ada hal lain yang bisa saya bantu terkait keranjang belanja Anda?
====== separator ======
>> Pemanggilan 3, tidak perlu tool apapun, cuma pertanyaan biasa <<
Halo! Saya adalah asisten belanja virtual An